# **Crypto Market Exploration**
---
### What do we actually want to understand?

- *What does BTC trade data look like?*
- *What is the mid-price doing over time?*
- *How wide is the spread?*
- *When is the market most active?*
- *Are there anomolies or weird patterns?*

## 1) Get BTC Data
---

### Load Libraries

In [53]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

print("Libraries loaded successfully")

Libraries loaded successfully


Get Bitcoin trade data from Coinbase.

In [54]:
def get_btc_trades(limit=1000):
    url = "https://api.exchange.coinbase.com/products/BTC-USD/trades"
    params = {"limit": limit}
    response = requests.get(url, params=params)
    data = response.json()
    df = pd.DataFrame(data)
    return df

trades = get_btc_trades()
print(f"Got {len(trades)} trades")


Got 1000 trades


In [55]:
trades.head()

,trade_id,side,size,price,time
0,1017363569,buy,0.00005944,81100.14000000,2026-05-13T01:49:38.306037Z
1,1017363568,buy,0.00017299,81100.14000000,2026-05-13T01:49:38.306037Z
2,1017363567,sell,0.00141094,81100.15000000,2026-05-13T01:49:34.371077Z
3,1017363566,sell,0.00093606,81100.15000000,2026-05-13T01:49:34.371077Z
4,1017363565,sell,0.00001400,81100.15000000,2026-05-13T01:49:34.371077Z


In [56]:
trades['price'] = pd.to_numeric(trades['price'])
trades['size'] = pd.to_numeric(trades['size'])
trades['time'] = pd.to_datetime(trades['time'])

trades = trades.sort_values('time').reset_index(drop=True)
trades = trades.set_index("time")


print(trades.dtypes)
trades.head()

trade_id      int64
side         object
size        float64
price       float64
dtype: object


,trade_id,side,size,price
time,,,,
2026-05-13 01:43:55.804673+00:00,1017362570,sell,3.654100e-04,81113.91
2026-05-13 01:43:55.804673+00:00,1017362571,sell,1.066725e-02,81113.91
2026-05-13 01:43:55.850441+00:00,1017362572,sell,3.040500e-04,81113.91
2026-05-13 01:43:56.274241+00:00,1017362573,buy,4.000000e-08,81113.90
2026-05-13 01:43:56.451906+00:00,1017362575,buy,1.479430e-02,81112.32


## 2) Calculate mid-price over time
---

### Noise vs. lag tradeoff (will adjust this parameter later)

In [57]:
trades['mid_price'] = trades['price'].rolling(window=20).mean()
# trades = trades.sort_index(ascending=False)
trades.tail(25)

,trade_id,side,size,price,mid_price
time,,,,,
2026-05-13 01:49:17.111401+00:00,1017363545,sell,5.029500e-04,81096.80,81096.5310
2026-05-13 01:49:18.819986+00:00,1017363546,sell,2.930000e-06,81096.80,81096.6985
2026-05-13 01:49:20.613461+00:00,1017363547,sell,1.258240e-03,81096.80,81096.7960
2026-05-13 01:49:23.478624+00:00,1017363548,buy,6.955200e-04,81096.79,81096.7955
2026-05-13 01:49:23.478624+00:00,1017363549,buy,1.948200e-04,81096.79,81096.7950
2026-05-13 01:49:23.478624+00:00,1017363550,buy,4.280000e-04,81096.79,81096.7950
2026-05-13 01:49:24.323625+00:00,1017363551,sell,1.169610e-03,81096.80,81096.7955
2026-05-13 01:49:25.476614+00:00,1017363552,sell,3.860000e-05,81096.80,81096.7960
2026-05-13 01:49:26.949552+00:00,1017363553,sell,1.082430e-03,81096.80,81096.7965


In [58]:
trades['side_switch'] = trades['side'] != trades['side'].shift(1)
switches = trades[trades['side_switch']]

In [60]:
trades['price_diff'] = switches['price'].diff().abs()
trades['spread_estimate'] = trades['price_diff'].rolling(window=20).mean()
trades.tail(10)

,trade_id,side,size,price,mid_price,side_switch,price_diff,spread_estimate
time,,,,,,,,
2026-05-13 01:49:29.349423+00:00,1017363560,sell,1.169610e-03,81098.02,81096.8585,False,NaN,NaN
2026-05-13 01:49:29.693172+00:00,1017363561,sell,1.479600e-02,81099.24,81096.9805,False,NaN,NaN
2026-05-13 01:49:30.118545+00:00,1017363562,buy,6.000000e-08,81100.14,81097.1475,True,3.34,NaN
2026-05-13 01:49:31.160929+00:00,1017363563,buy,6.000000e-08,81100.14,81097.3145,False,NaN,NaN
2026-05-13 01:49:32.219831+00:00,1017363564,buy,4.000000e-08,81100.14,81097.4815,False,NaN,NaN
2026-05-13 01:49:34.371077+00:00,1017363565,sell,1.400000e-05,81100.15,81097.6490,True,0.01,NaN
2026-05-13 01:49:34.371077+00:00,1017363566,sell,9.360600e-04,81100.15,81097.8165,False,0.01,NaN
2026-05-13 01:49:34.371077+00:00,1017363567,sell,1.410940e-03,81100.15,81097.9840,False,0.01,NaN
2026-05-13 01:49:38.306037+00:00,1017363568,buy,1.729900e-04,81100.14,81098.1515,True,0.01,NaN


In [69]:
switches = trades[trades['side_switch']]
switches.tail()

,trade_id,side,size,price,mid_price,side_switch,price_diff,spread_estimate
time,,,,,,,,
2026-05-13 01:49:27.437729+00:00,1017363555,buy,1.689000e-05,81096.79,81096.7960,True,0.01,NaN
2026-05-13 01:49:29.286749+00:00,1017363557,sell,6.863527e-02,81096.80,81096.7965,True,0.01,NaN
2026-05-13 01:49:30.118545+00:00,1017363562,buy,6.000000e-08,81100.14,81097.1475,True,3.34,NaN
2026-05-13 01:49:34.371077+00:00,1017363565,sell,1.400000e-05,81100.15,81097.6490,True,0.01,NaN
2026-05-13 01:49:38.306037+00:00,1017363568,buy,1.729900e-04,81100.14,81098.1515,True,0.01,NaN


In [67]:
print(f"Average estimated spread: ${switches['price_diff'].mean():.2f}")
print(f"Median estimated spread: ${switches['price_diff'].mean():.2f}")
print(f"Min: ${switches['price_diff'].min():.2f}")
print(f"Max: ${switches['price_diff'].max():.2f}")

Average estimated spread: $1.72
Median estimated spread: $1.72
Min: $0.01
Max: $18.55


- **$1.72 average spread** on a BTC price of roughly $60,000 means the spread is about 0.003% of the price. That's extremely tight — which tells you BTC is a very liquid, heavily competed market. Lots of market makers fighting for the same edge.

- **$0.01 minimum** — that's essentially zero. Those are trades happening at nearly identical prices, back to back. High activity, lots of competition.

- **$18.55 maximum** — that's a spread blowout. Something caused liquidity to temporarily disappear. Could be a news event, a large order wiping out the book, or just a thin moment. This is exactly the kind of environment where a naive market maker gets hurt badly.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

buys = trades[trades['side'] == 'buy']
sells = trades[trades['side'] == 'sell']

ax.scatter(buys.index, buys['price'], color='green', s=4, alpha=0.4, label='Buy')
ax.scatter(sells.index, sells['price'], color='red', s=4, alpha=0.4, label='Sell')

ax.plot(trades.index, trades['mid_price'], color='royalblue', linewidth=1.5, label='Mid-price (rolling 20)')

ax.set_title('BTC-USD Trades with Mid-Price Rolling Average')
ax.set_xlabel('Time (UTC)')
ax.set_ylabel('Price (USD)')
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()